## Simple Evaluation

Evaluate the RAG system's answer quality and performance.

> **About Evaluation:** Testing your RAG system is critical for ensuring answer quality. This notebook implements simple evaluation metrics: answer relevance, source accuracy, response time, and token usage. These metrics help identify issues and track improvements.

### Import Libraries

In [1]:
# Core libraries
from llama_index.llms.groq import Groq
from llama_index.core import Settings

# Vector database and embeddings
import chromadb
from sentence_transformers import SentenceTransformer

# Environment and utilities
from dotenv import load_dotenv
import os
import time
import json
from pathlib import Path
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import APIStatusError

### Configuration

In [2]:
# Global constants
MODEL_NAME = "all-MiniLM-L6-v2"  # Embedding model
GROQ_MODEL = "llama-3.1-8b-instant"  # LLM model (best for free tier)
CONTEXT_WINDOW = 131072  # 128K context support
MAX_CONTEXT_LENGTH = 15000  # Characters to stay within 6K TPM limit
TOP_K_RESULTS = 3  # Number of chunks to retrieve

# ChromaDB collection name
COLLECTION_NAME = "legal-docs"

### Load Environment Variables

In [3]:
# Load API keys from .env file
load_dotenv()

# Get Groq API key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY or GROQ_API_KEY == "your_groq_api_key_here":
    print("⚠ Warning: GROQ_API_KEY not set!")
    print("Please create a .env file with your API key.")
    print("See .env.example for template.")
else:
    print("✓ GROQ_API_KEY loaded successfully")
    print(f"  Key starts with: {GROQ_API_KEY[:8]}...")

✓ GROQ_API_KEY loaded successfully
  Key starts with: gsk_Q5Ti...


### Initialize Components

In [4]:
# Initialize embedding model (load once, reuse everywhere)
print(f"Loading embedding model: {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
print("✓ Embedding model loaded")

# Initialize Groq LLM
print(f"Initializing Groq LLM: {GROQ_MODEL}...")
llm = Groq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
    context_window=CONTEXT_WINDOW
)
Settings.llm = llm
print("✓ Groq LLM initialized")

# Connect to ChromaDB (running in Docker)
print("Connecting to ChromaDB (localhost:8000)...")
chroma_client = chromadb.HttpClient(host="localhost", port="8000")
print(f"✓ ChromaDB connected: {chroma_client.heartbeat()}")

# Get or create collection
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)
print(f"✓ Collection ready: {collection.name}")
print(f"  Total documents: {collection.count()}")

Loading embedding model: all-MiniLM-L6-v2...
✓ Embedding model loaded
Initializing Groq LLM: llama-3.1-8b-instant...
✓ Groq LLM initialized
Connecting to ChromaDB (localhost:8000)...
✓ ChromaDB connected: 1773820434593898757
✓ Collection ready: legal-docs
  Total documents: 57


### Helper Functions

In [5]:
def estimate_tokens(text):
    """
    Rough estimate of tokens in text (4 chars ≈ 1 token for English).
    
    Args:
        text: Input text string
    
    Returns:
        Estimated token count
    """
    return len(text) // 4


def limit_context(context, max_length=MAX_CONTEXT_LENGTH, verbose=True):
    """
    Limit context length to avoid Groq API token rate limits.
    
    Args:
        context: The full context string from retrieved documents
        max_length: Maximum character limit (default: 15,000 chars ≈ 3,750 tokens)
        verbose: Whether to print warning when truncating
    
    Returns:
        Truncated context string with notification if truncated
    
    Note:
        Groq free tier has 6,000 TPM (tokens per minute) limit for llama-3.1-8b-instant.
        Keep context under 4,000 tokens to leave room for prompt + response.
    """
    if len(context) > max_length:
        if verbose:
            est_before = estimate_tokens(context)
            est_after = estimate_tokens(context[:max_length])
            print(f"⚠ Context truncated from {len(context):,} to {max_length:,} chars")
            print(f"  (est. tokens: {est_before:,} → {est_after:,} to fit 6K TPM limit)")
        return context[:max_length] + "\n\n[Context truncated to fit token limits]"
    return context


def build_context_from_results(results, max_length=MAX_CONTEXT_LENGTH):
    """
    Build context from ChromaDB query results with automatic length limiting.
    
    Args:
        results: ChromaDB query results object
        max_length: Maximum character limit for final context
    
    Returns:
        Tuple of (context_string, num_docs)
    """
    context_docs = results["documents"][0]
    context = "\n\n".join(context_docs)
    context = limit_context(context, max_length)
    return context, len(context_docs)


def is_rate_limit_error(exception):
    """Check if exception is a rate limit error (429 or 413)."""
    if isinstance(exception, APIStatusError):
        return exception.status_code in [429, 413]
    return False


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=5, max=60),
    retry=retry_if_exception_type(APIStatusError),
    reraise=True
)
def llm_complete_with_retry(llm, prompt):
    """
    Call LLM completion with retry logic for rate limit errors.
    
    Args:
        llm: Groq LLM instance
        prompt: Input prompt
    
    Returns:
        LLM completion response
    
    Note:
        Retries on 429 (rate limit) and 413 (request too large) errors
        with exponential backoff: 5s, 10s, 20s, etc.
    """
    try:
        return llm.complete(prompt)
    except APIStatusError as e:
        if is_rate_limit_error(e):
            print(f"⚠ Rate limit hit: {e}")
            print(f"  Retrying with exponential backoff...")
        raise

### RAG Prompt Template

In [6]:
# Constrained RAG prompt for accuracy (from Notebook 07)
CONSTRAINED_RAG_PROMPT = """
You are a precise legal assistant. Answer questions using ONLY the provided context.

Context:
{context}

Question: {question}

Instructions:
1. Answer ONLY based on the provided context - do not use external knowledge
2. Cite specific sections, paragraphs, or sources when available
3. If the answer is not in the context, state "I don't have enough information in the provided context"
4. Do not make assumptions or inferences beyond what is explicitly stated
5. Keep responses concise but complete

Answer:
"""

### Evaluation Functions

In [7]:
def ask_question(query, top_k=TOP_K_RESULTS, verbose=False):
    """
    Ask a question and get an answer from the RAG pipeline.
    
    Args:
        query: User's question string
        top_k: Number of chunks to retrieve from ChromaDB (default: 3)
        verbose: Whether to print intermediate steps
    
    Returns:
        dict with keys:
            - 'query': Original question
            - 'answer': The LLM-generated answer
            - 'sources': List of source document names
            - 'response_time': Time taken in seconds
            - 'chunks_used': Number of chunks retrieved
            - 'context_length': Length of context in characters
            - 'estimated_tokens': Estimated tokens used
    """
    start_time = time.time()
    
    # Step 1: Generate query embedding
    query_embedding = model.encode([query]).tolist()
    
    # Step 2: Retrieve from ChromaDB
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )
    
    # Check if we got results
    if not results["documents"][0]:
        return {
            "query": query,
            "answer": "I don't have enough information in the provided context to answer this question.",
            "sources": [],
            "response_time": time.time() - start_time,
            "chunks_used": 0,
            "context_length": 0,
            "estimated_tokens": 0
        }
    
    # Step 3: Build context
    context, num_docs = build_context_from_results(results, MAX_CONTEXT_LENGTH)
    
    # Extract source document names
    sources = list(set(
        meta.get("file_name", "unknown") 
        for meta in results["metadatas"][0]
    ))
    
    # Step 4: Format prompt
    formatted_prompt = CONSTRAINED_RAG_PROMPT.format(
        context=context,
        question=query
    )
    
    # Estimate tokens
    est_tokens = estimate_tokens(formatted_prompt)
    
    # Step 5: Call Groq LLM
    response = llm_complete_with_retry(llm, formatted_prompt)
    
    # Calculate response time
    response_time = time.time() - start_time
    
    return {
        "query": query,
        "answer": str(response),
        "sources": sources,
        "response_time": response_time,
        "chunks_used": num_docs,
        "context_length": len(context),
        "estimated_tokens": est_tokens
    }

In [8]:
def evaluate_answer_relevance(answer, query, context):
    """
    Evaluate answer relevance on a 1-5 scale using LLM judgment.
    
    Args:
        answer: The generated answer
        query: The original question
        context: The context used to generate the answer
    
    Returns:
        dict with keys:
            - 'score': Relevance score (1-5)
            - 'reason': Explanation for the score
    """
    evaluation_prompt = f"""
You are an evaluator of RAG system answers. Rate the answer's relevance to the question.

Question: {query}

Answer: {answer}

Rate the answer on a scale of 1-5:
1 = Completely irrelevant or off-topic
2 = Mostly irrelevant with some relevant points
3 = Partially relevant but misses key points
4 = Mostly relevant and addresses the question well
5 = Perfectly relevant, complete, and accurate

Provide your rating as a JSON object:
{{
    "score": <1-5>,
    "reason": "<brief explanation>"
}}

Response:"""
    
    try:
        response = llm_complete_with_retry(llm, evaluation_prompt)
        # Parse JSON response
        response_str = str(response).strip()
        # Extract JSON from response (handle markdown code blocks)
        if "```json" in response_str:
            response_str = response_str.split("```json")[1].split("```")[0].strip()
        elif "```" in response_str:
            response_str = response_str.split("```")[1].split("```")[0].strip()
        
        evaluation = json.loads(response_str)
        return {
            "score": evaluation.get("score", 0),
            "reason": evaluation.get("reason", "No reason provided")
        }
    except Exception as e:
        return {
            "score": 0,
            "reason": f"Evaluation failed: {str(e)}"
        }

In [9]:
def verify_source_accuracy(answer, sources):
    """
    Verify if the answer correctly cites the provided sources.
    
    Args:
        answer: The generated answer
        sources: List of source document names
    
    Returns:
        dict with keys:
            - 'citations_found': Boolean indicating if citations were present
            - 'sources_cited': List of sources actually cited in the answer
            - 'accuracy': Boolean indicating if citations match provided sources
    """
    answer_lower = answer.lower()
    sources_cited = []
    
    # Check which sources are mentioned in the answer
    for source in sources:
        # Extract base filename without extension for matching
        base_name = Path(source).stem.lower()
        if base_name in answer_lower or source.lower() in answer_lower:
            sources_cited.append(source)
    
    # Check for generic citation phrases
    citations_found = any(phrase in answer_lower for phrase in [
        "according to",
        "as stated",
        "as mentioned",
        "the document",
        "the context",
        "section",
        "para",
        "paragraph"
    ])
    
    # Accuracy: if sources were cited, they should match the provided sources
    accuracy = len(sources_cited) == 0 or all(
        any(Path(s).stem.lower() == Path(c).stem.lower() for s in sources)
        for c in sources_cited
    )
    
    return {
        "citations_found": citations_found or len(sources_cited) > 0,
        "sources_cited": sources_cited,
        "accuracy": accuracy
    }

In [10]:
def run_evaluation(query, verbose=True):
    """
    Run complete evaluation for a single query.
    
    Args:
        query: The question to evaluate
        verbose: Whether to print detailed results
    
    Returns:
        dict with complete evaluation results
    """
    if verbose:
        print(f"\n{'=' * 60}")
        print(f"Query: {query}")
        print('=' * 60)
    
    # Step 1: Get answer from RAG pipeline
    start_time = time.time()
    result = ask_question(query, verbose=False)
    pipeline_time = time.time() - start_time
    
    if verbose:
        print(f"\n✓ Answer generated in {result['response_time']:.2f}s")
        print(f"\nAnswer:\n{result['answer']}")
        print(f"\nSources: {result['sources']}")
    
    # Step 2: Evaluate relevance
    if verbose:
        print("\nEvaluating answer relevance...")
    relevance_result = evaluate_answer_relevance(
        result['answer'], 
        query, 
        "Context used"  # Simplified for evaluation
    )
    if verbose:
        print(f"  Relevance Score: {relevance_result['score']}/5")
        print(f"  Reason: {relevance_result['reason']}")
    
    # Step 3: Verify source accuracy
    if verbose:
        print("\nVerifying source accuracy...")
    accuracy_result = verify_source_accuracy(
        result['answer'], 
        result['sources']
    )
    if verbose:
        print(f"  Citations found: {accuracy_result['citations_found']}")
        print(f"  Sources cited: {accuracy_result['sources_cited']}")
        print(f"  Accuracy: {'✓' if accuracy_result['accuracy'] else '✗'}")
    
    # Compile complete evaluation
    evaluation = {
        "query": query,
        "answer": result['answer'],
        "sources": result['sources'],
        "metrics": {
            "response_time_seconds": round(result['response_time'], 2),
            "chunks_used": result['chunks_used'],
            "context_length_chars": result['context_length'],
            "estimated_tokens": result['estimated_tokens'],
            "relevance_score": relevance_result['score'],
            "relevance_reason": relevance_result['reason'],
            "citations_found": accuracy_result['citations_found'],
            "sources_cited": accuracy_result['sources_cited'],
            "source_accuracy": accuracy_result['accuracy']
        }
    }
    
    if verbose:
        print(f"\n{'=' * 60}")
        print("EVALUATION SUMMARY")
        print('=' * 60)
        print(f"Response Time: {result['response_time']:.2f}s")
        print(f"Relevance: {relevance_result['score']}/5")
        print(f"Source Accuracy: {'✓' if accuracy_result['accuracy'] else '✗'}")
    
    return evaluation

### Test Queries for Evaluation

In [11]:
# Define test queries covering different aspects of the legal documents
TEST_QUERIES = [
    "What are the forest conservation issues?",
    "What is the penalty for violating the Forest Conservation Act?",
    "What approval is needed to use forest land for non-forest purposes?",
    "What is the constitutional basis for forest protection in India?",
    "What did the Supreme Court say about tiger reserves?"
]

print(f"Prepared {len(TEST_QUERIES)} test queries for evaluation")
print("\nTest queries:")
for i, query in enumerate(TEST_QUERIES, 1):
    print(f"  {i}. {query}")

Prepared 5 test queries for evaluation

Test queries:
  1. What are the forest conservation issues?
  2. What is the penalty for violating the Forest Conservation Act?
  3. What approval is needed to use forest land for non-forest purposes?
  4. What is the constitutional basis for forest protection in India?
  5. What did the Supreme Court say about tiger reserves?


### Run Evaluation on All Test Queries

In [12]:
# Run evaluation for all test queries
all_evaluations = []

print(f"Running evaluation on {len(TEST_QUERIES)} queries...")
print("This may take a few minutes due to API calls.\n")

for i, query in enumerate(TEST_QUERIES, 1):
    print(f"\n[{i}/{len(TEST_QUERIES)}] Evaluating query {i}...")
    evaluation = run_evaluation(query, verbose=True)
    all_evaluations.append(evaluation)

Running evaluation on 5 queries...
This may take a few minutes due to API calls.


[1/5] Evaluating query 1...

Query: What are the forest conservation issues?

✓ Answer generated in 1.87s

Answer:
According to the provided context, the forest conservation issues mentioned are:

1. Destruction of forests due to agriculture (Paragraph 1).
2. Depletion of forests due to rapid urbanization, unchecked industrialization, encroachments, etc. (Paragraph 3).
3. Insufficient forest cover in India, which is about 21.76% of the total landmass (Paragraph 4).
4. Encroachments on forest areas, with almost 13,000 sq. km of forests under encroachment (Paragraph 5).
5. Need to transform from an anthropocentric approach to an ecocentric approach for the interest of the environment (Paragraph 7).

Additionally, the context mentions the following specific issues:

- The exploitation of the Singampatti Zamin forest lands in Tamil Nadu, where the lease holders cleared out the forests and started cultivating

### Aggregate Evaluation Results

In [13]:
# Calculate aggregate statistics
total_queries = len(all_evaluations)
avg_response_time = sum(e['metrics']['response_time_seconds'] for e in all_evaluations) / total_queries
avg_relevance_score = sum(e['metrics']['relevance_score'] for e in all_evaluations) / total_queries
avg_chunks_used = sum(e['metrics']['chunks_used'] for e in all_evaluations) / total_queries
avg_tokens = sum(e['metrics']['estimated_tokens'] for e in all_evaluations) / total_queries

citations_found_count = sum(1 for e in all_evaluations if e['metrics']['citations_found'])
source_accuracy_count = sum(1 for e in all_evaluations if e['metrics']['source_accuracy'])

print("\n" + "=" * 60)
print("AGGREGATE EVALUATION RESULTS")
print("=" * 60)
print(f"\nTotal queries evaluated: {total_queries}")
print(f"\nPerformance Metrics:")
print(f"  Average response time: {avg_response_time:.2f} seconds")
print(f"  Average chunks used: {avg_chunks_used:.1f}")
print(f"  Average tokens per query: {avg_tokens:.0f}")
print(f"\nQuality Metrics:")
print(f"  Average relevance score: {avg_relevance_score:.2f}/5.00")
print(f"  Queries with citations: {citations_found_count}/{total_queries} ({citations_found_count/total_queries*100:.0f}%)")
print(f"  Source accuracy: {source_accuracy_count}/{total_queries} ({source_accuracy_count/total_queries*100:.0f}%)")
print("\n" + "=" * 60)


AGGREGATE EVALUATION RESULTS

Total queries evaluated: 5

Performance Metrics:
  Average response time: 4.01 seconds
  Average chunks used: 3.0
  Average tokens per query: 1481

Quality Metrics:
  Average relevance score: 3.60/5.00
  Queries with citations: 4/5 (80%)
  Source accuracy: 5/5 (100%)



### Detailed Results Table

In [14]:
# Display detailed results in table format
print("\nDETAILED RESULTS TABLE")
print("=" * 120)
print(f"{'#':<3} {'Query (truncated)':<40} {'Time (s)':<10} {'Relevance':<12} {'Citations':<12} {'Accuracy':<10}")
print("-" * 120)

for i, eval_result in enumerate(all_evaluations, 1):
    query_preview = eval_result['query'][:37] + "..." if len(eval_result['query']) > 40 else eval_result['query']
    time_str = f"{eval_result['metrics']['response_time_seconds']:.2f}"
    relevance_str = f"{eval_result['metrics']['relevance_score']}/5"
    citations_str = "Yes" if eval_result['metrics']['citations_found'] else "No"
    accuracy_str = "✓" if eval_result['metrics']['source_accuracy'] else "✗"
    
    print(f"{i:<3} {query_preview:<40} {time_str:<10} {relevance_str:<12} {citations_str:<12} {accuracy_str:<10}")

print("=" * 120)


DETAILED RESULTS TABLE
#   Query (truncated)                        Time (s)   Relevance    Citations    Accuracy  
------------------------------------------------------------------------------------------------------------------------
1   What are the forest conservation issues? 1.87       4/5          Yes          ✓         
2   What is the penalty for violating the... 0.31       4/5          No           ✓         
3   What approval is needed to use forest... 0.33       5/5          Yes          ✓         
4   What is the constitutional basis for ... 1.51       2/5          Yes          ✓         
5   What did the Supreme Court say about ... 16.05      3/5          Yes          ✓         


### Export Evaluation Results

In [15]:
# Export results to JSON file
from datetime import datetime

# Create export data
export_data = {
    "evaluation_date": datetime.now().isoformat(),
    "total_queries": total_queries,
    "aggregate_metrics": {
        "avg_response_time_seconds": round(avg_response_time, 2),
        "avg_relevance_score": round(avg_relevance_score, 2),
        "avg_chunks_used": round(avg_chunks_used, 1),
        "avg_estimated_tokens": round(avg_tokens, 0),
        "citations_found_percentage": round(citations_found_count / total_queries * 100, 0),
        "source_accuracy_percentage": round(source_accuracy_count / total_queries * 100, 0)
    },
    "individual_evaluations": all_evaluations
}

# Save to file
OUTPUT_DIR = Path("../data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_file = OUTPUT_DIR / f"evaluation_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"

with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(export_data, f, indent=2, ensure_ascii=False)

print(f"✓ Evaluation results exported to: {output_file}")
print(f"\nExported data includes:")
print(f"  - Aggregate metrics summary")
print(f"  - {total_queries} individual evaluation records")
print(f"  - Full answers and sources for each query")

✓ Evaluation results exported to: ../data/evaluation_results_20260318_132422.json

Exported data includes:
  - Aggregate metrics summary
  - 5 individual evaluation records
  - Full answers and sources for each query


### Quality Assessment and Recommendations

In [16]:
# Generate quality assessment based on metrics
print("\nQUALITY ASSESSMENT")
print("=" * 60)

# Relevance assessment
if avg_relevance_score >= 4.0:
    print("\n✓ RELEVANCE: EXCELLENT")
    print(f"  Average score {avg_relevance_score:.2f}/5 indicates high-quality answers")
elif avg_relevance_score >= 3.0:
    print("\n⚠ RELEVANCE: GOOD")
    print(f"  Average score {avg_relevance_score:.2f}/5 is acceptable but has room for improvement")
else:
    print("\n✗ RELEVANCE: NEEDS IMPROVEMENT")
    print(f"  Average score {avg_relevance_score:.2f}/5 suggests answers are not sufficiently relevant")

# Citation assessment
citation_rate = citations_found_count / total_queries
if citation_rate >= 0.8:
    print("\n✓ CITATIONS: EXCELLENT")
    print(f"  {citation_rate*100:.0f}% of answers include proper citations")
elif citation_rate >= 0.5:
    print("\n⚠ CITATIONS: GOOD")
    print(f"  {citation_rate*100:.0f}% of answers include citations - could be improved")
else:
    print("\n✗ CITATIONS: NEEDS IMPROVEMENT")
    print(f"  Only {citation_rate*100:.0f}% of answers include citations")

# Source accuracy assessment
accuracy_rate = source_accuracy_count / total_queries
if accuracy_rate >= 0.9:
    print("\n✓ SOURCE ACCURACY: EXCELLENT")
    print(f"  {accuracy_rate*100:.0f}% accuracy in source attribution")
elif accuracy_rate >= 0.7:
    print("\n⚠ SOURCE ACCURACY: GOOD")
    print(f"  {accuracy_rate*100:.0f}% accuracy - minor improvements needed")
else:
    print("\n✗ SOURCE ACCURACY: NEEDS IMPROVEMENT")
    print(f"  {accuracy_rate*100:.0f}% accuracy suggests issues with source attribution")

# Performance assessment
if avg_response_time <= 3.0:
    print("\n✓ PERFORMANCE: EXCELLENT")
    print(f"  Average response time {avg_response_time:.2f}s is very good")
elif avg_response_time <= 5.0:
    print("\n⚠ PERFORMANCE: GOOD")
    print(f"  Average response time {avg_response_time:.2f}s is acceptable")
else:
    print("\n✗ PERFORMANCE: NEEDS IMPROVEMENT")
    print(f"  Average response time {avg_response_time:.2f}s is slow - consider optimization")

print("\n" + "=" * 60)


QUALITY ASSESSMENT

⚠ RELEVANCE: GOOD
  Average score 3.60/5 is acceptable but has room for improvement

✓ CITATIONS: EXCELLENT
  80% of answers include proper citations

✓ SOURCE ACCURACY: EXCELLENT
  100% accuracy in source attribution

⚠ PERFORMANCE: GOOD
  Average response time 4.01s is acceptable



### Recommendations for Improvement

In [17]:
print("\nRECOMMENDATIONS")
print("=" * 60)

recommendations = []

# Generate recommendations based on metrics
if avg_relevance_score < 4.0:
    recommendations.append(
        "1. IMPROVE RELEVANCE:\n"
        "   - Try different prompt templates (see Notebook 07)\n"
        "   - Increase top_k results to retrieve more context\n"
        "   - Use chain-of-thought prompting for complex queries"
    )

if citation_rate < 0.8:
    recommendations.append(
        "2. IMPROVE CITATIONS:\n"
        "   - Modify prompt to explicitly request citations\n"
        "   - Add source metadata to chunks for better attribution\n"
        "   - Use few-shot examples showing proper citation format"
    )

if avg_response_time > 3.0:
    recommendations.append(
        "3. OPTIMIZE PERFORMANCE:\n"
        "   - Reduce context length (currently using 15K chars)\n"
        "   - Use fewer chunks (reduce top_k from 3 to 2)\n"
        "   - Consider using llama-3.2-11b-vision-preview for faster responses"
    )

if accuracy_rate < 0.9:
    recommendations.append(
        "4. IMPROVE SOURCE ACCURACY:\n"
        "   - Add more specific citation instructions to prompt\n"
        "   - Include page numbers or section references in metadata\n"
        "   - Verify retrieved chunks match the query intent"
    )

if not recommendations:
    print("\n✓ All metrics are within excellent ranges!")
    print("  Consider running more comprehensive tests with additional queries.")
else:
    for rec in recommendations:
        print(f"\n{rec}")

print("\n" + "=" * 60)


RECOMMENDATIONS

1. IMPROVE RELEVANCE:
   - Try different prompt templates (see Notebook 07)
   - Increase top_k results to retrieve more context
   - Use chain-of-thought prompting for complex queries

3. OPTIMIZE PERFORMANCE:
   - Reduce context length (currently using 15K chars)
   - Use fewer chunks (reduce top_k from 3 to 2)
   - Consider using llama-3.2-11b-vision-preview for faster responses



### Summary

In [18]:
print("\nEVALUATION COMPLETE")
print("=" * 60)
print(f"\nEvaluated {total_queries} queries with the following results:\n")
print(f"  ✓ Average Relevance Score: {avg_relevance_score:.2f}/5.00")
print(f"  ✓ Average Response Time: {avg_response_time:.2f} seconds")
print(f"  ✓ Citation Rate: {citation_rate*100:.0f}%")
print(f"  ✓ Source Accuracy: {accuracy_rate*100:.0f}%")
print(f"\nResults saved to: {output_file.name}")
print("\nNext Steps:")
print("  - Review individual evaluations for specific issues")
print("  - Apply recommendations to improve RAG pipeline")
print("  - Re-run evaluation to measure improvements")
print("  - Expand test queries for more comprehensive evaluation")
print("=" * 60)


EVALUATION COMPLETE

Evaluated 5 queries with the following results:

  ✓ Average Relevance Score: 3.60/5.00
  ✓ Average Response Time: 4.01 seconds
  ✓ Citation Rate: 80%
  ✓ Source Accuracy: 100%

Results saved to: evaluation_results_20260318_132422.json

Next Steps:
  - Review individual evaluations for specific issues
  - Apply recommendations to improve RAG pipeline
  - Re-run evaluation to measure improvements
  - Expand test queries for more comprehensive evaluation
